# 04 Chroma — 벡터 DB

numpy 검색은 벡터를 메모리에 다 올린다. Chroma 는 같은 코사인 검색을 디스크에 맡기고, `metadata` 로 문서를 걸러 준다. 이미 만든 `chunks.jsonl` 과 `vectors.npy` 를 그대로 넣는다.


In [ ]:
from pathlib import Path  # 경로를 문자열 대신 객체로 다룬다
import os  # 환경변수(OPENAI_API_KEY)를 넣기 위해 쓴다

# 수업 코드는 키가 이미 있는 상태를 가정한다. 이 노트북은 .env를 직접 읽는다.
for _env in (Path("../.env"), Path("../../c3-api/.env")):  # 프로젝트 루트, 옆 폴더 순으로 찾는다
    if not _env.is_file():  # 파일이 없으면 다음 후보
        continue  # 있는 파일만 읽는다
    for _line in _env.read_text(encoding="utf-8").splitlines():  # .env를 한 줄씩
        _line = _line.strip()  # 앞뒤 공백 제거
        if not _line or _line.startswith("#") or "=" not in _line:  # 빈 줄·주석·형식 아닌 줄
            continue  # 건너뛴다
        _k, _v = _line.split("=", 1)  # KEY=VALUE 로 나눈다
        os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))  # 이미 있으면 덮지 않는다


In [ ]:
import json  # 조각 파일 읽기
import shutil  # 옛 chroma 폴더를 지울 때
from pathlib import Path  # 경로

import chromadb  # 벡터 DB
import numpy as np  # vectors.npy
from openai import OpenAI  # 질문도 같은 모델로 임베딩해야 한다

client = OpenAI()  # 질문 임베딩용
EMBED_MODEL = "text-embedding-3-small"  # 03 과 반드시 같은 모델

chunks = [json.loads(l) for l in Path("chunks.jsonl").read_text(encoding="utf-8").splitlines()]  # 조각
V = np.load("vectors.npy")  # 03 에서 저장한 좌표. 행 번호 = 조각 번호
print(f"조각 {len(chunks):,}개 · 벡터 {V.shape}")  # 두 길이가 같아야 넣을 수 있다


In [ ]:
CHROMA_DIR = Path("chroma")  # day02/chroma . 수업 체크리스트: 폴더를 지우고 다시 만든다
if CHROMA_DIR.exists():  # 어제 인덱스가 남아 있으면 답이 이상해진다
    shutil.rmtree(CHROMA_DIR)  # 폴더 통째로 삭제
    print("지움", CHROMA_DIR)  # 확인

chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))  # 디스크에 남는 클라이언트
col = chroma.create_collection(  # 새 컬렉션
    name="corpus",  # 이름. 나중에 get_collection 으로 다시 연다
    metadata={"hnsw:space": "cosine"},  # 거리 정의를 코사인으로. numpy 검색과 맞춘다
)

BATCH = 100  # add 도 나눠 보낸다


def clean_meta(meta):  # Chroma 는 값으로 문자열·숫자·참거짓만 받는다
    out = {}  # 깨끗한 사전
    for k, v in meta.items():  # 키마다
        if v is None:  # 없는 헤더(h3 가 없는 조각)
            continue  # 키 자체를 빼 둔다
        out[str(k)] = v if isinstance(v, (str, int, float, bool)) else str(v)  # 나머지는 문자열로
    return out  # 컬렉션이 받을 수 있는 형태


for i in range(0, len(chunks), BATCH):  # 0, 100, 200, …
    batch = chunks[i : i + BATCH]  # 이번 묶음
    col.add(  # 벡터 DB 에 삽입
        ids=[c["metadata"]["chunk_id"] for c in batch],  # 고유 이름. 같은 id 를 두 번 넣지 않는다
        documents=[c["text"] for c in batch],  # 나중에 그대로 돌려 줄 본문
        metadatas=[clean_meta(c["metadata"]) for c in batch],  # doc_id, source, h1, h2, h3
        embeddings=V[i : i + BATCH].tolist(),  # 이미 계산한 벡터. 다시 API 를 부르지 않는다
    )
    print(f"  {min(i + BATCH, len(chunks)):>4}/{len(chunks)}")  # 진행

print("컬렉션", col.count())  # 넣은 개수 = 조각 수


In [ ]:
def search_chroma(question, k=3, where=None):  # numpy search 와 같은 모양
    q = client.embeddings.create(model=EMBED_MODEL, input=[question]).data[0].embedding  # 질문 좌표
    res = col.query(  # DB 에게 가까운 조각을 물어본다
        query_embeddings=[q],  # 리스트로 한 개
        n_results=k,  # 상위 k
        where=where,  # None 이면 전체. {"doc_id": "pipa"} 면 그 문서만
        include=["documents", "metadatas", "distances"],  # 본문·출처·거리
    )
    hits = []  # (본문, metadata, 거리)
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):  # 첫 질문의 결과
        hits.append((doc, meta, float(dist)))  # 거리는 작을수록 가깝다. numpy 점수와 스케일이 다르다
    return hits  # 순위만 본다


QUESTION = "어린이 정보를 다룰 때 지켜야 할 것은?"  # 03 과 같은 질문
print("질문:", QUESTION)  # 제목
for text, m, dist in search_chroma(QUESTION):  # 가까운 순
    print(f"{dist:.3f}  {m['chunk_id']:20s} {m.get('h3') or m['doc_id']}")  # 거리, 아이디, 소제목
    print(f"        {text[:90].strip()}…")  # 본문 앞
    print()  # 빈 줄


metadata 필터로 검색 범위를 한 문서로 좁힌다. 어제 장부의 `doc_id` 가 여기로 이어진다.


In [ ]:
print("--- pipa 만 ---")  # 개인정보 보호법 조각만
for text, m, dist in search_chroma("개인정보처리자의 정의는?", k=3, where={"doc_id": "pipa"}):  # where 가 필터
    print(f"{dist:.3f}  {m['chunk_id']}  {m.get('h3') or ''}")  # 법 조항 제목이 h3 에 있으면 찍힌다
    print(f"        {text[:80].strip()}…")  # 본문
    print()  # 빈 줄

print("--- paper-wheat-dss 만 ---")  # 밀 논문만
for text, m, dist in search_chroma("과정기반 작물모형 이름은?", k=3, where={"doc_id": "paper-wheat-dss"}):  # 문서 제한
    print(f"{dist:.3f}  {m['chunk_id']}")  # 아이디
    print(f"        {text[:80].strip()}…")  # 본문
    print()  # 빈 줄

print("점수의 절대 기준은 없다. 같은 질문 안에서의 순위만 의미가 있다.")  # 지식 목표 한 줄


In [ ]:
# 다음 시간에 쓸 수 있게 컬렉션을 다시 열어 보기만 한다
again = chromadb.PersistentClient(path=str(CHROMA_DIR))  # 같은 폴더
col2 = again.get_collection("corpus")  # 이름로 기존 컬렉션
print("다시 연 개수", col2.count())  # 커널을 꺼도 디스크에 남아 있어야 한다
